In [0]:
%pip install prophet

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyspark.sql.functions as F
import datetime as datetime
from prophet import Prophet


In [0]:
events = spark.table("events_silver").toPandas()
print(events.head())
events.info()

In [0]:
events.event_time = pd.to_datetime(events.event_time)
events.info()



In [0]:
events_group = events.groupby("event")["year"].count()
events_group.head(90)


In [0]:
df = spark.sql("SELECT * FROM retail_rocket.db.ts_purchases").toPandas()

In [0]:
df.plot(figsize=(15, 5))

In [0]:
df["rolling_mean"] = df["purchases"].rolling(window=7).mean()

import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))
plt.plot(df["date"], df["purchases"], label="Original", alpha=0.5)
plt.plot(df["date"], df["rolling_mean"], label="Trend (7 days)", linewidth=2)
plt.legend()
plt.xticks(rotation=45)
plt.show()


In [0]:
df["date"] = pd.to_datetime(df["date"])
df["day_of_week"] = df["date"].dt.day_name()
sns.boxplot(x="day_of_week", y="purchases", data=df)
plt.xticks(rotation=45)

In [0]:
df_prophet = df[["date", "purchases"]].rename(columns={
    "date": "ds",
    "purchases": "y"
})


In [0]:
model = Prophet()
model.fit(df_prophet)

In [0]:
future = model.make_future_dataframe(periods=30)
forecast = model.predict(future)

fig1 = model.plot(forecast)
plt.show()

fig2 = model.plot_components(forecast)
plt.show()


In [0]:
plt.figure(figsize=(15,5))

plt.plot(df_prophet["ds"], df_prophet["y"], label="Real")
plt.plot(forecast["ds"], forecast["yhat"], label="Forecast")

plt.legend()
plt.title("Forecast vs Real")
plt.xticks(rotation=45)

plt.show()